# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My Rule (The CTR-Fix Logic):
If a page ranks on the first page of Google (Average Position <= 10) and gets a solid amount of visibility (Impressions > 500), but has an unusually low Click-Through Rate (CTR < 1%), it indicates that the title or meta description is failing to attract users.
**Action:** Rewrite Title & Meta.

Reason Codes:
* `HIGH_VISIBILITY_POOR_CTR`: Impressions > 500, Position <= 10, CTR < 0.01 (Score: 100)
* `MODERATE_VISIBILITY_POOR_CTR`: Impressions between 100-500, Position <= 10, CTR < 0.01 (Score: 50)
* `NO_ACTION`: CTR is healthy or visibility is too low to judge (Score: 0)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import pandas as pd
import os
from google.colab import userdata

# Setup output directory (CI leak-guard respects this directory)
os.makedirs('work/outputs', exist_ok=True)
hf_token = userdata.get('HF_TOKEN')

print("Fetching performance table from data warehouse...")
# Fact table from previous week
fact_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
df_fact = pd.read_parquet(fact_path, storage_options={"token": hf_token})

# Aggregate daily data to page level
df = df_fact.groupby(['client_hash_id', 'content_hash_id']).agg({
    'gsc_impressions': 'sum',
    'gsc_clicks': 'sum',
    'gsc_avg_position': 'mean'
}).reset_index()

# Calculate Click-Through Rate (CTR) safely (prevent division by zero)
df['ctr'] = df['gsc_clicks'] / df['gsc_impressions'].replace(0, 1)

# Function implementing the manual baseline rule
def score_page(row):
    if row['gsc_avg_position'] <= 10 and row['ctr'] < 0.01:
        if row['gsc_impressions'] > 500:
            return pd.Series([100, 'HIGH_VISIBILITY_POOR_CTR', 'Rewrite Title/Meta'])
        elif row['gsc_impressions'] > 100:
            return pd.Series([50, 'MODERATE_VISIBILITY_POOR_CTR', 'Review Title/Meta'])
    return pd.Series([0, 'NO_ACTION', 'None'])

df[['baseline_score', 'reason_code', 'action_label']] = df.apply(score_page, axis=1)

# Filter only actionable items and sort by score and impressions
queue_df = df[df['baseline_score'] > 0].sort_values(
    by=['baseline_score', 'gsc_impressions'],
    ascending=[False, False]
)

# Write results to CSV (will remain local, excluded from git)
csv_path = 'work/outputs/baseline_action_score.csv'
queue_df.to_csv(csv_path, index=False)

print(f"Pipeline finished! Found {len(queue_df)} actionable pages and saved to {csv_path}.")
print("\n--- TOP 5 PREVIEW ---")
print(queue_df[['content_hash_id', 'gsc_impressions', 'ctr', 'gsc_avg_position', 'action_label']].head())

Fetching performance table from data warehouse...
Pipeline finished! Found 52410 actionable pages and saved to work/outputs/baseline_action_score.csv.

--- TOP 5 PREVIEW ---
                 content_hash_id  gsc_impressions       ctr  gsc_avg_position  \
305394  content_eadb33b5df496f4a           617124  0.009185          2.383011   
305436  content_ec2e0346994fb5a5           245276  0.006034          2.854514   
297401  content_0e03de7680314cd5           221310  0.003253          2.675217   
60074   content_44f34c0a90047651           212404  0.000113          7.346909   
186197  content_7172a7fad43f0998           205867  0.004187          3.367835   

              action_label  
305394  Rewrite Title/Meta  
305436  Rewrite Title/Meta  
297401  Rewrite Title/Meta  
60074   Rewrite Title/Meta  
186197  Rewrite Title/Meta  


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 Review (Skeptic's Eye):

1-5. Action: Rewrite Title/Meta | Reason: HIGH_VISIBILITY_POOR_CTR
* Confidence: High. These pages have massive impressions on Page 1 but almost zero clicks.
* What would make it wrong: Zero-click searches. If the user searched for something like "weather today", Google shows the answer directly. The user gets it without clicking our link. Rewriting the title won't fix this structural SEO reality.

6-10. Action: Rewrite Title/Meta | Reason: HIGH_VISIBILITY_POOR_CTR
* Confidence: Moderate-High.
* What would make it wrong: Brand mismatch. The page might be unintentionally ranking for a competitor's brand name. Users see our link, realize we aren't that brand, and scroll past.

11-20. Action: Rewrite Title/Meta | Reason: HIGH_VISIBILITY_POOR_CTR
* Confidence: Moderate.
* What would make it wrong: Image Pack/Video Carousel dominance. Our text link might technically be "Position 8", but it's physically buried under massive image widgets. The CTR is low because of Google's UI layout, not our text.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks Analysis:
The weakest picks generated by this rule are the pages ranking exactly at position 9.5 to 10.0. Technically they are "Page 1", but practically they are at the very bottom of the SERP, where natural CTR is already extremely low (often below 1% organically). Flagging them for a title rewrite might be a waste of resources; they actually need link-building to move up to position 1-3 first.

Leakage Check:
Confirmed clean. The rule strictly uses historical `gsc_impressions`, `gsc_clicks`, and `gsc_avg_position` from the March 2026 dataset. No future windows were observed, and no pre-calculated labels or target data (like the trend_pct we excluded last week) leaked into this baseline scoring logic.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.